# Bloque 3: Carga y Feature Stores
## Módulo ETCDFM · Unidad Didáctica 3

**Extracción y carga desde bases de datos relacionales**

---

### Contenidos
1. DDL: `CREATE TABLE`, tipos, constraints, índices, vistas
2. DML: `INSERT`, `UPDATE`, `DELETE`
3. `df.to_sql`: parámetros y estrategias
4. Carga masiva: `executemany` y rendimiento
5. OLTP vs OLAP: modelos de datos destino
6. DCL: roles y permisos
7. Feature stores: concepto y Feast local
8. Pipeline ETL completo con SQLAlchemy


## DDL: Data Definition Language

DDL define la **estructura** de la base de datos.

```sql
CREATE TABLE ventas_fact (
    id_venta    INTEGER PRIMARY KEY AUTOINCREMENT,
    id_cliente  INTEGER NOT NULL REFERENCES clientes(id_cliente),
    id_producto INTEGER NOT NULL REFERENCES productos(id_producto),
    fecha       TEXT    NOT NULL,
    importe     REAL    NOT NULL CHECK (importe >= 0),
    canal       TEXT    DEFAULT 'web'
);

CREATE INDEX idx_vf_fecha   ON ventas_fact(fecha);
CREATE INDEX idx_vf_cliente ON ventas_fact(id_cliente);

CREATE VIEW v_resumen_diario AS
    SELECT fecha, COUNT(*) AS n_ventas, SUM(importe) AS total
    FROM   ventas_fact
    GROUP BY fecha;

ALTER TABLE ventas_fact ADD COLUMN descuento REAL DEFAULT 0.0;
DROP TABLE ventas_fact;  -- irreversible!
```

**Tipos de datos SQL (SQLite es flexible; PostgreSQL es estricto):**

| Tipo | SQLite | PostgreSQL |
|---|---|---|
| Entero | `INTEGER` | `INTEGER`, `BIGINT`, `SMALLINT` |
| Decimal | `REAL` | `NUMERIC(p,s)`, `FLOAT`, `DOUBLE PRECISION` |
| Texto | `TEXT` | `VARCHAR(n)`, `CHAR(n)`, `TEXT` |
| Fecha | `TEXT` (ISO 8601) | `DATE`, `TIMESTAMP`, `TIMESTAMPTZ` |
| Booleano | `INTEGER` (0/1) | `BOOLEAN` |


In [ ]:
import sqlite3
import pandas as pd
import os, shutil

# Crear una BD nueva para los demos de carga
TMP_DB = '/tmp/demo_carga.db'
if os.path.exists(TMP_DB):
    os.remove(TMP_DB)

con = sqlite3.connect(TMP_DB)
cur = con.cursor()
cur.execute('PRAGMA foreign_keys = ON')

# DDL: crear schema de destino
cur.executescript("""
CREATE TABLE dim_categoria (
    id_categoria  INTEGER PRIMARY KEY,
    nombre        TEXT NOT NULL UNIQUE,
    activo        INTEGER NOT NULL DEFAULT 1
);

CREATE TABLE dim_tiempo (
    fecha         TEXT PRIMARY KEY,   -- YYYY-MM-DD
    anno          INTEGER NOT NULL,
    mes           INTEGER NOT NULL,
    trimestre     INTEGER NOT NULL,
    dia_semana    INTEGER NOT NULL,   -- 0=lunes .. 6=domingo
    es_fin_semana INTEGER NOT NULL DEFAULT 0
);

CREATE TABLE fact_ventas (
    id_venta      INTEGER PRIMARY KEY AUTOINCREMENT,
    fecha         TEXT NOT NULL REFERENCES dim_tiempo(fecha),
    id_categoria  INTEGER NOT NULL REFERENCES dim_categoria(id_categoria),
    n_pedidos     INTEGER NOT NULL CHECK (n_pedidos >= 0),
    importe_total REAL    NOT NULL CHECK (importe_total >= 0),
    ticket_medio  REAL    NOT NULL
);

CREATE INDEX idx_fv_fecha ON fact_ventas(fecha);
CREATE INDEX idx_fv_cat  ON fact_ventas(id_categoria);

CREATE VIEW v_ventas_mensuales AS
    SELECT t.anno, t.mes,
           SUM(f.n_pedidos)     AS pedidos_mes,
           ROUND(SUM(f.importe_total),2) AS importe_mes
    FROM   fact_ventas f
    JOIN   dim_tiempo t ON f.fecha = t.fecha
    GROUP BY t.anno, t.mes;
""")

# Listar objetos creados
cur.execute("SELECT type, name FROM sqlite_master WHERE type IN ('table','view','index') ORDER BY type, name")
print('Objetos creados en demo_carga.db:')
for row in cur.fetchall():
    print(f'  [{row[0]:<5}] {row[1]}')
con.close()


## DML: Data Manipulation Language

DML manipula los **datos** dentro de la estructura creada con DDL.

```sql
-- INSERT: una fila
INSERT INTO productos (nombre, precio, stock)
VALUES ('Nuevo producto', 99.99, 50);

-- INSERT desde otra tabla (ETL clásico)
INSERT INTO fact_ventas (fecha, id_categoria, importe_total)
SELECT fecha, id_categoria, SUM(importe)
FROM   staging_ventas
GROUP BY fecha, id_categoria;

-- UPDATE: modificar filas existentes
UPDATE productos
SET    precio = precio * 1.05   -- subida del 5%
WHERE  id_categoria = 1
  AND  stock > 0;

-- DELETE: eliminar filas
DELETE FROM pedidos
WHERE  estado = 'cancelado'
  AND  fecha < '2023-01-01';

-- UPSERT (SQLite >= 3.24): insertar o actualizar si ya existe
INSERT INTO productos (id_producto, nombre, precio)
VALUES (1, 'Smart TV', 499.99)
ON CONFLICT (id_producto) DO UPDATE SET precio = excluded.precio;
```

> **TRANSACCIONES:** `BEGIN`, `COMMIT`, `ROLLBACK` garantizan atomicidad (ACID).


In [ ]:
con = sqlite3.connect(TMP_DB)
cur = con.cursor()
cur.execute('PRAGMA foreign_keys = ON')

# INSERT categorias
categorias = [
    (1,'Electronica'),(2,'Informatica'),(3,'Moviles'),
    (4,'Ropa'),(5,'Hogar'),(6,'Deportes'),(7,'Libros'),(8,'Alimentacion'),
]
cur.executemany('INSERT INTO dim_categoria VALUES (?,?,1)', categorias)
print(f'dim_categoria: {cur.rowcount} filas insertadas')

# INSERT dim_tiempo (365 dias de 2024)
from datetime import date, timedelta
dias = []
d = date(2024, 1, 1)
for _ in range(365):
    ds = d.strftime('%Y-%m-%d')
    dias.append((ds, d.year, d.month, (d.month-1)//3+1,
                 d.weekday(), 1 if d.weekday() >= 5 else 0))
    d += timedelta(days=1)
cur.executemany('INSERT INTO dim_tiempo VALUES (?,?,?,?,?,?)', dias)
print(f'dim_tiempo: {len(dias)} filas insertadas')

# UPDATE: marcar como inactiva la categoria Alimentacion
cur.execute("UPDATE dim_categoria SET activo = 0 WHERE nombre = 'Alimentacion'")
print(f'UPDATE: {cur.rowcount} fila(s) modificada(s)')

# DELETE con transaccion explicita
con.commit()
cur.execute('BEGIN')
try:
    cur.execute("DELETE FROM dim_categoria WHERE activo = 0")
    n = cur.rowcount
    con.commit()
    print(f'DELETE (con COMMIT): {n} categoria(s) eliminada(s)')
except Exception as e:
    con.rollback()
    print(f'ROLLBACK por error: {e}')

# Volver a insertar Alimentacion
cur.execute("INSERT INTO dim_categoria VALUES (8,'Alimentacion',1)")
con.commit()
cur.execute('SELECT COUNT(*) FROM dim_categoria')
print(f'dim_categoria final: {cur.fetchone()[0]} filas')
con.close()


## `df.to_sql`: Carga de DataFrames en BD

```python
df.to_sql(
    name      = 'tabla_destino',
    con       = engine,             # SQLAlchemy engine o conexion sqlite3
    if_exists = 'append',           # 'fail' | 'replace' | 'append'
    index     = False,              # no escribir el indice de pandas como columna
    chunksize = 1000,               # insertar en lotes
    dtype     = {'fecha': Text()},  # forzar tipos SQLAlchemy
    method    = 'multi'             # INSERT multi-fila (mas rapido)
)
```

| `if_exists=` | Comportamiento |
|---|---|
| `'fail'` | Error si la tabla ya existe (por defecto) |
| `'replace'` | DROP + CREATE + INSERT (¡pierde datos existentes!) |
| `'append'` | Solo INSERT (respeta datos existentes) |

**Tipos SQLAlchemy para `dtype=`:**
```python
from sqlalchemy import Integer, Float, Text, DateTime, Boolean
dtype = {'id': Integer(), 'nombre': Text(), 'precio': Float()}
```

> `method='multi'` genera un único `INSERT` con múltiples filas por lote
> → hasta **10× más rápido** que inserciones individuales.


In [ ]:
from sqlalchemy import create_engine, Text, Integer, Float
import time

engine = create_engine(f'sqlite:///{TMP_DB}')

# Cargar ventas_diarias.csv y preparar datos
ventas = pd.read_csv('datos/ventas_diarias.csv', parse_dates=['fecha'])
ventas['fecha_str'] = ventas['fecha'].dt.strftime('%Y-%m-%d')

# Filtrar solo categorias activas (1-8) y fechas en dim_tiempo (2024)
ventas_2024 = ventas[ventas['fecha'].dt.year == 2024].copy()
print(f'Filas a cargar: {len(ventas_2024)}')

# to_sql: carga en fact_ventas
fact = ventas_2024[['fecha_str','id_categoria','n_pedidos','importe_total','ticket_medio']]\
       .rename(columns={'fecha_str': 'fecha'})

# Comparar rendimiento: sin 'multi' vs con 'multi'
t0 = time.perf_counter()
fact.to_sql('fact_ventas', engine, if_exists='replace', index=False,
            chunksize=100)
t1 = time.perf_counter() - t0

t0 = time.perf_counter()
fact.to_sql('fact_ventas', engine, if_exists='replace', index=False,
            chunksize=100, method='multi')
t2 = time.perf_counter() - t0

print(f'to_sql sin method=multi: {t1*1000:.1f} ms')
print(f'to_sql con method=multi: {t2*1000:.1f} ms')
print(f'Mejora: {t1/t2:.1f}x')

# Verificar via vista
df_vista = pd.read_sql('SELECT * FROM v_ventas_mensuales ORDER BY anno, mes LIMIT 6', engine)
print('\nVista v_ventas_mensuales (primeros 6 meses):')
print(df_vista.to_string(index=False))


## Carga Masiva: Estrategias de Rendimiento

Para millones de filas, el rendimiento es crítico.

| Estrategia | Velocidad | Uso |
|---|---|---|
| `INSERT` individual | 🐌 Lenta | Filas únicas, upserts |
| `executemany()` | 🐇 Rápida | Lotes Python → BD |
| `df.to_sql(method='multi')` | 🐇 Rápida | DataFrames |
| `COPY` (PostgreSQL) | 🚀 Muy rápida | Ficheros planos masivos |
| Desactivar índices + INSERT + recrear | 🚀 Muy rápida | Carga inicial masiva |

**`executemany` con SQLite:**
```python
con.executemany(
    'INSERT INTO tabla VALUES (?,?,?)',
    lista_de_tuplas         # iterable de filas
)
```

**`COPY` con PostgreSQL (psycopg2):**
```python
import io
buf = io.StringIO(df.to_csv(index=False, header=False))
cur.copy_expert(
    'COPY tabla (col1, col2, col3) FROM STDIN WITH CSV',
    buf
)
```

> Con PostgreSQL, `COPY` es **5-50× más rápido** que `INSERT` para grandes volúmenes.


In [ ]:
import time

# Cargar productos_nuevos.csv y comparar estrategias de insercion
prods = pd.read_csv('datos/productos_nuevos.csv')
print(f'Productos a insertar: {len(prods)}')

con_tmp = sqlite3.connect(TMP_DB)
cur = con_tmp.cursor()

# Crear tabla staging
cur.execute('DROP TABLE IF EXISTS staging_productos')
cur.execute("""
    CREATE TABLE staging_productos (
        id_producto INTEGER, nombre TEXT, id_categoria INTEGER,
        precio REAL, coste REAL, stock INTEGER, proveedor TEXT, activo INTEGER
    )
""")

filas = list(prods.itertuples(index=False, name=None))
N = 500  # Repetir la lista para aumentar el volumen
filas_grandes = [(row[0]+i*1000,)+row[1:] for i in range(N) for row in filas]

# Estrategia 1: INSERT individual (sin transaccion)
cur.execute('DELETE FROM staging_productos')
t0 = time.perf_counter()
for row in filas_grandes[:1000]:  # solo 1000 para no tardar mucho
    cur.execute('INSERT INTO staging_productos VALUES (?,?,?,?,?,?,?,?)', row)
con_tmp.commit()
t1 = time.perf_counter() - t0
print(f'INSERT individual (1000 filas): {t1*1000:.1f} ms')

# Estrategia 2: executemany con transaccion
cur.execute('DELETE FROM staging_productos')
t0 = time.perf_counter()
cur.executemany('INSERT INTO staging_productos VALUES (?,?,?,?,?,?,?,?)', filas_grandes)
con_tmp.commit()
t2 = time.perf_counter() - t0
print(f'executemany ({len(filas_grandes):,} filas): {t2*1000:.1f} ms')
print(f'Mejora: {t1/t2*(len(filas_grandes)/1000):.0f}x (ajustado por volumen)')

cur.execute('SELECT COUNT(*) FROM staging_productos')
print(f'Filas en staging: {cur.fetchone()[0]:,}')
con_tmp.close()


## OLTP vs OLAP: Modelos de Datos Destino

| | **OLTP** | **OLAP** |
|---|---|---|
| Propósito | Operaciones transaccionales | Analítica y reporting |
| Diseño | 3NF normalizado | Star schema / Snowflake |
| Operaciones | INSERT/UPDATE/DELETE frecuentes | Lecturas masivas |
| Filas por consulta | Pocas (por id) | Millones (agregaciones) |
| Índices | Muchos | Pocos (columnar storage) |
| Ejemplo | Base de datos de pedidos | Data warehouse de ventas |

### Star Schema (esquema estrella)
```
        dim_tiempo
             |
dim_cliente -- fact_ventas -- dim_producto
             |
        dim_categoria
```
- **Tabla de hechos** (`fact_*`): métricas numéricas + claves foráneas
- **Dimensiones** (`dim_*`): atributos descriptivos (ciudad, categoría, fecha...)

> En ML, los **feature stores** siguen una lógica similar:
> entidades (dim) + features (hechos con timestamp).


![Esquema en estrella para el análisis de ventas](imagenes/b3_star_schema.png)

## DCL: Data Control Language

DCL gestiona **permisos y seguridad** de acceso.

```sql
-- PostgreSQL: crear roles y usuarios
CREATE ROLE analista;                         -- rol sin login
CREATE USER etl_user WITH PASSWORD 's3cr3t'; -- usuario con login

-- Conceder permisos
GRANT SELECT ON ALL TABLES IN SCHEMA public TO analista;
GRANT SELECT, INSERT ON fact_ventas TO etl_user;
GRANT USAGE ON SEQUENCE fact_ventas_id_seq TO etl_user;

-- Asignar rol a usuario
GRANT analista TO etl_user;

-- Revocar permisos
REVOKE INSERT ON fact_ventas FROM etl_user;

-- Row Level Security (PostgreSQL)
ALTER TABLE clientes ENABLE ROW LEVEL SECURITY;
CREATE POLICY p_region ON clientes
    USING (ciudad = current_setting('app.ciudad'));
```

> SQLite no implementa DCL (es un fichero local — los permisos son del SO).
> En PostgreSQL/SQL Server, DCL es fundamental para producción.

**Principio de mínimo privilegio:** cada usuario/proceso solo tiene los
permisos estrictamente necesarios para su función.


## Feature Stores: ¿Qué son y por qué importan?

Un **feature store** es un sistema centralizado para almacenar, versionar
y servir **features de ML** de forma consistente entre entrenamiento e inferencia.

### Problema que resuelven
```
❌ Sin feature store:
  Equipo A calcula 'gasto_promedio' como media del último mes
  Equipo B calcula 'gasto_promedio' como media de los últimos 3 meses
  → El modelo entrenado con A falla en producción con B

✅ Con feature store:
  Una sola definición → misma feature en entrenamiento Y en producción
```

### Componentes
- **Offline store**: datos históricos para entrenamiento (Parquet, Delta Lake, BigQuery)
- **Online store**: acceso en baja latencia para inferencia (Redis, DynamoDB, SQLite)
- **Registry**: definición y versión de entidades y features

### Herramientas
| Herramienta | Tipo | Notable por |
|---|---|---|
| **Feast** | Open source | Standard de la industria, portabilidad |
| **Hopsworks** | Open source / Cloud | Feature monitoring, Great Expectations |
| **Tecton** | Comercial | Streaming features, SLA garantizado |
| **Vertex AI Feature Store** | Cloud (GCP) | Integración con BigQuery |


![Arquitectura de un feature store](imagenes/b3_feature_store.png)

In [ ]:
# Feature store simplificado con SQLite + pandas
# (En produccion: Feast + Redis/BigQuery)

FEATURE_STORE_DB = '/tmp/feature_store.db'
if os.path.exists(FEATURE_STORE_DB):
    os.remove(FEATURE_STORE_DB)

# 1. Definir y cargar features
features = pd.read_csv('datos/features_clientes.csv', parse_dates=['feature_timestamp'])
print('Features disponibles:', list(features.columns))
print(f'Shape: {features.shape}')

# 2. Guardar en el feature store (SQLite como online store)
fs_engine = create_engine(f'sqlite:///{FEATURE_STORE_DB}')
features.to_sql('client_features', fs_engine, if_exists='replace', index=False,
                method='multi')
print(f'\nFeatures cargadas en el store: {FEATURE_STORE_DB}')

# 3. Recuperar features por entidad (punto de vista de inferencia)
def get_features(id_clientes: list) -> pd.DataFrame:
    ids = ','.join(str(i) for i in id_clientes)
    return pd.read_sql(
        f'SELECT * FROM client_features WHERE id_cliente IN ({ids})',
        fs_engine
    )

# Simular peticion de inferencia para 5 clientes
batch = get_features([1, 5, 10, 42, 77])
print('\nFeatures para batch de inferencia:')
print(batch[['id_cliente','recencia_dias','frecuencia','monetario',
             'ticket_medio','pct_cancelados']].to_string(index=False))

# 4. Point-in-time join (concepto clave de feature stores)
print('\n--- Point-in-time join ---')
print('En un feature store real, cada feature tiene un timestamp.')
print('El PIT join garantiza que al entrenar NO se usan features')
print('calculadas DESPUES del evento que queremos predecir (data leakage).')


## Pipeline ETL Completo con SQLAlchemy

Integrando todo lo aprendido en UD3:

```
1. EXTRACT  ─── SQL → pd.read_sql (con pushdown de filtros)
2. TRANSFORM ── pandas (limpieza, feature engineering, pivot)
3. LOAD     ─── df.to_sql (carga en BD destino OLAP)
```

**Buenas prácticas:**
- Usar **tablas staging** como buffer antes de la tabla destino final
- Implementar **idempotencia**: el pipeline puede ejecutarse varias veces sin duplicar datos
- Logging de cada etapa: filas extraídas, transformadas, cargadas
- Manejo de errores con `try/except/rollback`
- Carga **incremental** cuando sea posible (filtrar por `fecha_modificacion`)

```python
def run_etl(source_engine, dest_engine, fecha_desde):
    # Extract
    df_raw = pd.read_sql(SQL_EXTRACT, source_engine,
                         params={'fecha': fecha_desde})
    # Transform
    df_clean = transform(df_raw)
    # Load
    df_clean.to_sql('fact_tabla', dest_engine,
                    if_exists='append', method='multi')
    return len(df_clean)
```


In [ ]:
import logging
from datetime import datetime

logging.basicConfig(level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S')
log = logging.getLogger('ETL')

# Source: tienda.db (OLTP)
src_db = os.path.abspath(os.path.join('datos', 'tienda.db'))

if not os.path.isfile(src_db):
    raise FileNotFoundError(f'No se encuentra la BD fuente: {src_db}')

src_engine = create_engine(f"sqlite:///{src_db.replace(os.sep, '/')}")
# Destination: demo_carga.db (OLAP star schema)
dst_engine = create_engine(f'sqlite:///{TMP_DB}')

def extract(engine, fecha_desde='2023-01-01'):
    sql = f"""
        SELECT
            p.fecha,
            cat.id_categoria,
            COUNT(DISTINCT p.id_pedido)               AS n_pedidos,
            ROUND(SUM(d.cantidad * d.precio_unit), 2) AS importe_total,
            ROUND(AVG(d.precio_unit), 2)               AS ticket_medio
        FROM   pedidos p
        JOIN   detalles_pedido d ON p.id_pedido   = d.id_pedido
        JOIN   productos pr      ON d.id_producto = pr.id_producto
        JOIN   categorias cat    ON pr.id_categoria = cat.id_categoria
        WHERE  p.estado != 'cancelado'
          AND  p.fecha >= '{fecha_desde}'
        GROUP BY p.fecha, cat.id_categoria
    """
    return pd.read_sql(sql, engine, parse_dates=['fecha'])

def transform(df):
    df = df.copy()
    # Solo fechas en dim_tiempo (2024)
    df = df[df['fecha'].dt.year == 2024]
    df['fecha'] = df['fecha'].dt.strftime('%Y-%m-%d')
    df['n_pedidos'] = df['n_pedidos'].astype(int)
    return df[['fecha','id_categoria','n_pedidos','importe_total','ticket_medio']]

def load(df, engine):
    with engine.begin() as conn:
        conn.execute(engine.dialect.statement_compiler(
            engine.dialect, None).__class__  # dummy
        ) if False else None  # skip — usar to_sql directamente
    df.to_sql('fact_ventas', engine, if_exists='replace',
              index=False, method='multi')
    return len(df)

# Ejecutar pipeline
log.info('=== Inicio pipeline ETL ===')
t_start = datetime.now()

log.info('EXTRACT: consultando BD fuente...')
df_raw = extract(src_engine)
log.info(f'  {len(df_raw):,} filas extraidas')

log.info('TRANSFORM: limpiando y preparando datos...')
df_clean = transform(df_raw)
log.info(f'  {len(df_clean):,} filas tras transformacion')

log.info('LOAD: cargando en BD destino...')
n_loaded = len(df_clean)
df_clean.to_sql('fact_ventas', dst_engine, if_exists='replace',
                index=False, method='multi')
log.info(f'  {n_loaded:,} filas cargadas')

t_end = (datetime.now() - t_start).total_seconds()
log.info(f'=== Pipeline completado en {t_end:.2f}s ===')

# Verificar resultado final
resumen = pd.read_sql('SELECT * FROM v_ventas_mensuales ORDER BY anno, mes', dst_engine)
print('\nResumen mensual (vista OLAP):')
print(resumen.to_string(index=False))


## Resumen del Bloque 3

| Concepto | SQL | Python / pandas |
|---|---|---|
| Crear estructura | `CREATE TABLE`, `CREATE INDEX`, `CREATE VIEW` | `engine.execute(DDL)` |
| Insertar datos | `INSERT INTO ... VALUES` | `df.to_sql(if_exists='append')` |
| Actualizar datos | `UPDATE ... SET ... WHERE` | `con.execute(UPDATE)` |
| Eliminar datos | `DELETE FROM ... WHERE` | `con.execute(DELETE)` |
| Carga masiva | `COPY` (PG), `BULK INSERT` (SQL Server) | `executemany`, `method='multi'` |
| Permisos | `GRANT`, `REVOKE`, `CREATE ROLE` | — (DCL es solo SQL) |
| Feature store | — | `feast`, store local SQLite/Parquet |
| Pipeline | — | extract → transform → load |

---

## Resumen de UD3

**RA2 cubierto:** acceso, extracción, DQL básico y avanzado, SQLAlchemy, paralela (chunking)

**RA3 cubierto:** DDL, DML, DCL, OLTP/OLAP, carga masiva, feature stores, pipeline ETL

**Próxima unidad →** UD4: Bases de datos NoSQL (documentales, clave-valor, vectoriales).
